# Disaggregation

How to expand typical-period results back to the original time series length.

**Use case:** You aggregate a year of hourly data into 8 typical days, run an optimization on those 8 days, and then need the results mapped back to all 365 days.

`disaggregate()` does exactly this — it takes any DataFrame with the same `(cluster, timestep)` structure as `cluster_representatives` and expands it using the stored cluster assignments.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

import tsam
from tsam import ClusteringResult, SegmentConfig

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)

## Basic Disaggregation

Aggregate, then disaggregate the typical periods back to the full year. The result matches `.reconstructed` exactly.

In [ ]:
result = tsam.aggregate(raw, n_clusters=8)

print(f"Original:               {result.original.shape}")
print(f"Cluster representatives: {result.cluster_representatives.shape}")

expanded = result.disaggregate(result.cluster_representatives)
print(f"Disaggregated:          {expanded.shape}")
print(f"Matches .reconstructed: {expanded.equals(result.reconstructed)}")

## Disaggregating Arbitrary Data

The real value: disaggregate data that tsam didn't produce. Here we simulate optimization results — a "dispatch" column computed from the typical periods — and expand it back to the full year.

In [ ]:
# Simulate optimization: compute "dispatch" as a function of the typical periods
reps = result.cluster_representatives
dispatch = pd.DataFrame(
    {"Dispatch": reps["Load"] - 0.5 * reps["GHI"] - 0.3 * reps["Wind"]},
    index=reps.index,
)

print(f"Dispatch (typical periods): {dispatch.shape}")
dispatch.head()

In [ ]:
# Disaggregate back to full year
full_year_dispatch = result.disaggregate(dispatch)

print(f"Full year dispatch: {full_year_dispatch.shape}")

fig = px.line(full_year_dispatch, labels={"index": "Time", "value": "Dispatch"})
fig.update_layout(title="Disaggregated Dispatch Over Full Year", showlegend=False)
fig.show()

## Survives IO

Save the clustering to JSON, load it later, and disaggregate without the original `AggregationResult`.

In [ ]:
# Save clustering
result.clustering.to_json("clustering.json")

# Later: load and disaggregate
clustering = ClusteringResult.from_json("clustering.json")
full_year_from_disk = clustering.disaggregate(dispatch)

print(f"Shape: {full_year_from_disk.shape}")
print(
    f"Matches original disaggregation: {full_year_dispatch.values[:8760].tolist() == full_year_from_disk.values.tolist()}"
)

Note: `ClusteringResult.disaggregate()` returns an integer-indexed DataFrame (it doesn't have access to the original datetime index). `AggregationResult.disaggregate()` restores the datetime index automatically.

## Segmented Data

With segmentation, `cluster_representatives` has a `(cluster, segment, duration)` index. Disaggregation expands segments to full timesteps, placing values at the start of each segment and NaN elsewhere. Use `.ffill()` for a step function.

In [ ]:
result_seg = tsam.aggregate(raw, n_clusters=8, segments=SegmentConfig(n_segments=4))

print(f"Cluster representatives: {result_seg.cluster_representatives.shape}")
print(f"Index levels: {result_seg.cluster_representatives.index.names}")
result_seg.cluster_representatives.head(8)

In [ ]:
expanded_seg = result_seg.disaggregate(result_seg.cluster_representatives)

print(f"Disaggregated shape: {expanded_seg.shape}")
print(f"NaN count: {expanded_seg.isna().sum().sum()} (segment gaps)")
print(f"Non-NaN count: {expanded_seg.notna().sum().sum()} (segment starts)")

# Show a single day: values at segment starts, NaN in between
expanded_seg["GHI"].iloc[:24]

In [ ]:
# Forward-fill for a step function
filled = expanded_seg.ffill()

fig = px.line(
    pd.DataFrame(
        {"Original": result_seg.original["GHI"], "Disaggregated (ffill)": filled["GHI"]}
    ),
    labels={"index": "Time", "value": "GHI"},
).update_layout(
    title="Segmented Disaggregation: Original vs Reconstructed (first 2 weeks)"
)
fig.update_xaxes(range=[raw.index[0], raw.index[24 * 14]])
fig.show()

## Summary

```python
# Aggregate
result = tsam.aggregate(df, n_clusters=8)

# Run optimization on typical periods
optimized = my_optimizer(result.cluster_representatives)

# Expand back to full year (with datetime index)
full_year = result.disaggregate(optimized)

# Or via saved clustering (integer index)
clustering = ClusteringResult.from_json("clustering.json")
full_year = clustering.disaggregate(optimized)

# For segmented data: .ffill() gives a step function
full_year = result.disaggregate(segmented_data).ffill()
```